<a href="https://colab.research.google.com/github/traderjohnd/foundation-model-from-scratch/blob/notebook-02-corpus-construction/notebooks/02_tokenizer_training_and_corpus_construction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building a Foundation Model from Scratch
## Notebook 02 — Tokenizer Training & Corpus Construction

This notebook begins from the verified data contract established in **Notebook 01 — Data Preparation & Corpus Audit**. It independently reloads the immutable WikiText-103 revision, applies the same locked normalization and article-reconstruction logic through `src/data.py`, trains the project tokenizer from scratch, validates it, records its checksum, and constructs the exact 20,000,000-token model-training corpus.

Notebook 01 is a completed audit artifact. Notebook 02 must not depend on Notebook 01's in-memory state.

### Pipeline boundary

```text
Notebook 01: raw WikiText → verified normalized articles
src/data.py: canonical reusable loading/normalization/reconstruction logic
Notebook 02: normalized articles → tokenizer → exact 20M-token corpus
Notebook 03: tokenizer/corpus → Transformer architecture
```


### Locked inputs and constraints

- Dataset: `Salesforce/wikitext`, `wikitext-103-raw-v1`
- Immutable Hub revision: `b08601e04326c79dfdd32d625aee71d232d685c3`
- Tokenizer: byte-level BPE trained from scratch
- Vocabulary size: 16,384 total tokens, including registered special tokens
- Tokenizer-training text: full normalized official training split only
- Model-training corpus: exactly 20,000,000 tokenizer-produced tokens
- Sampling seed: 42
- Validation remains development-visible; test remains untouched until final evaluation
- Article boundaries and normalization must reproduce Notebook 01's verified 28,472 training documents and 60 validation documents before tokenizer work proceeds

Canonical references: `docs/PROJECT_CONTEXT.md` and `docs/DECISION_REGISTER.md`.

# Chunk 1 — Reproduce the audited corpus from shared source code

Before tokenizer design begins, this chunk proves that Notebook 02 can independently reproduce Notebook 01's audited corpus. The verified normalization and article-reconstruction implementation has been extracted into `src/data.py` without refactoring the core logic.

The notebook pins the **source-code revision** containing that extraction. This matters because the Hub dataset revision alone fixes the upstream text, while the source-code revision fixes the exact transformation applied to it.

## 1. Load the canonical data pipeline at a fixed source revision

A Colab notebook opened from GitHub does not automatically make the repository's `src/` package importable. We therefore clone the project repository, check out the exact commit that introduced the verified extraction, and add the repository root to Python's import path.

This is intentionally a source-code pin, not a dependency install. The goal is for a fresh runtime to reconstruct the same data pipeline without relying on Notebook 01 or on whatever happens to be at the tip of `main` later.

In [1]:
%pip install -q datasets "tokenizers==0.23.1"


In [2]:
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/traderjohnd/foundation-model-from-scratch.git"
REPO_DIR = Path("/content/foundation-model-from-scratch")
DATA_PIPELINE_REVISION = "7d300f14c812d9a1caf36aa9ec0568bee5b0f275"

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-q", DATA_PIPELINE_REVISION], check=True)

sys.path.insert(0, str(REPO_DIR))
print(f"Loaded project source revision: {DATA_PIPELINE_REVISION}")


Loaded project source revision: 7d300f14c812d9a1caf36aa9ec0568bee5b0f275


In [3]:
import pandas as pd

from src.data import (
    AUDIT_SPLITS,
    DATASET_CONFIG,
    DATASET_ID,
    DATASET_REVISION,
    EXPECTED_ARTICLE_COUNTS,
    load_pinned_wikitext,
    normalize_development_splits,
    reconstruct_development_articles,
    run_normalization_self_test,
)

pd.Series({
    "dataset": DATASET_ID,
    "config": DATASET_CONFIG,
    "dataset_revision": DATASET_REVISION,
    "source_revision": DATA_PIPELINE_REVISION,
    "development_splits": AUDIT_SPLITS,
})


,0
dataset,Salesforce/wikitext
config,wikitext-103-raw-v1
dataset_revision,b08601e04326c79dfdd32d625aee71d232d685c3
source_revision,7d300f14c812d9a1caf36aa9ec0568bee5b0f275
development_splits,"(train, validation)"


## 2. Re-run the locked normalization regression tests

Notebook 01 established 17 explicit normalization cases. Running the same cases through `src/data.py` checks that extraction did not silently change the transformation contract.

In [4]:
normalization_test_results = run_normalization_self_test()
normalization_tests = pd.DataFrame(normalization_test_results)

assert len(normalization_tests) == 17
assert normalization_tests["passed"].all()
print("✓ 17/17 normalization regression tests passed.")
normalization_tests


✓ 17/17 normalization regression tests passed.


,input,expected,actual,passed
0,well @-@ known,well-known,well-known,True
1,3 @.@ 5 million,3.5 million,3.5 million,True
2,"1 @,@ 000 people","1,000 people","1,000 people",True
3,( 1987 ),(1987),(1987),True
4,don 't,don't,don't,True
5,café 's tables,café's tables,café's tables,True
6,the players ' hopes,the players ' hopes,the players ' hopes,True
7,$ 3 @.@ 5 million,$3.5 million,$3.5 million,True
8,The meeting ran from 12 : 30 to 13 : 05 .,The meeting ran from 12:30 to 13:05.,The meeting ran from 12:30 to 13:05.,True
9,"the "" Nameless "", a penal unit","the ""Nameless"", a penal unit","the ""Nameless"", a penal unit",True


## 3. Reload the immutable WikiText-103 revision

The module loads the same pinned Hub commit used by Notebook 01 and hard-checks the official row counts. Loading the dataset does **not** authorize development-time inspection of test examples; only `train` and `validation` are transformed below.

In [5]:
raw_dataset = load_pinned_wikitext()

split_rows = pd.Series(
    {split_name: raw_dataset[split_name].num_rows for split_name in raw_dataset},
    name="rows",
)
split_rows


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-103-raw-v1/test-00000-of-00001.(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-103-raw-v1/test-00000-of-00001.(…): downloading bytes:           |  0.00B            

wikitext-103-raw-v1/train-00000-of-00002(…): reconstructing file:   0%|          |  0.00B /  157MB            

wikitext-103-raw-v1/train-00000-of-00002(…): downloading bytes:           |  0.00B            

...

wikitext-103-raw-v1/validation-00000-of-(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

,rows
test,4358
train,1801350
validation,3760


## 4. Normalize development-visible splits only

The same locked function is applied to the official training and validation splits. The test split is deliberately not normalized or inspected during development.

In [6]:
normalized_development = normalize_development_splits(raw_dataset)

assert set(normalized_development) == {"train", "validation"}
print("✓ Normalized only train and validation.")


Normalize train:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Normalize validation:   0%|          | 0/3760 [00:00<?, ? examples/s]

✓ Normalized only train and validation.


## 5. Reconstruct articles and enforce the audit contract

Article starts are detected from the **raw** rows using the locked level-1-heading-plus-blank-neighbors rule, while the stored article text comes from the normalized rows. This is the same distinction that resolved the boundary-count discrepancies in Notebook 01.

The hard assertions below are the handoff gate. Tokenizer work does not proceed unless the shared module independently reproduces **28,472 training documents and 60 validation documents**.

In [7]:
articles_by_split = reconstruct_development_articles(
    raw_dataset,
    normalized_development,
)

article_summary = pd.DataFrame(
    [
        {
            "split": split_name,
            "articles": len(articles),
            "expected": EXPECTED_ARTICLE_COUNTS[split_name],
            "characters": sum(len(article["text"]) for article in articles),
            "first_article_id": articles[0]["article_id"],
            "last_article_id": articles[-1]["article_id"],
        }
        for split_name, articles in articles_by_split.items()
    ]
).set_index("split")

assert article_summary.loc["train", "articles"] == 28_472
assert article_summary.loc["validation", "articles"] == 60
print("✓ Shared pipeline reproduced the audited article counts.")
article_summary


✓ Shared pipeline reproduced the audited article counts.


,articles,expected,characters,first_article_id,last_article_id
split,,,,,
train,28472,28472,519078250,train:article-00000:row-1,train:article-28471:row-1801326
validation,60,60,1102768,validation:article-00000:row-1,validation:article-00059:row-3714


### Chunk 1 checkpoint — passed

A fresh runtime reloaded the pinned upstream corpus, executed the canonical shared preprocessing implementation, reran all 17 normalization tests, and reproduced the audited **28,472 training / 60 validation** article counts. Notebook 02 can therefore proceed to tokenizer design without depending on Notebook 01 kernel state.

# Chunk 2 — Lock the document-boundary / EOS special-token contract

Before BPE training begins, we need to decide which symbols are **structural** rather than learned from ordinary text. The project will use **`<|endoftext|>`** as the single registered special token. It serves as the document-boundary / end-of-sequence marker that will later be appended explicitly after each complete article.

This is a vocabulary contract, not tokenizer training. No BPE merges are learned in this chunk.

## 6. Define the structural vocabulary contract

The total vocabulary remains **16,384 IDs**, and the special token is reserved **inside** that total. We register no PAD, BOS, or UNK token.

- **No PAD:** the training pipeline will use fixed-length token sequences rather than padded variable-length examples.
- **No BOS:** the project does not need a separate beginning-of-sequence marker; document separation is handled by the boundary/EOS token.
- **No UNK:** byte-level BPE is designed to retain byte coverage, so an unknown-token fallback is unnecessary. Byte coverage will be explicitly validated after tokenizer training.
- **One boundary/EOS token:** `<|endoftext|>` is inserted by corpus construction, not automatically by ordinary text encoding.

Because the byte-level alphabet contains 256 byte symbols, a final 16,384-token vocabulary that reaches its requested size would contain 1 special token, 256 byte-alphabet tokens, and up to 16,127 merge-created vocabulary entries. We will verify the actual trained vocabulary rather than assuming the trainer reaches this exact composition.

In [8]:
VOCAB_SIZE = 16_384
DOC_BOUNDARY_TOKEN = "<|endoftext|>"
SPECIAL_TOKENS = [DOC_BOUNDARY_TOKEN]
BYTE_ALPHABET_SIZE = 256

assert len(SPECIAL_TOKENS) == 1
assert VOCAB_SIZE > BYTE_ALPHABET_SIZE + len(SPECIAL_TOKENS)

vocab_contract = pd.Series({
    "total_vocab_size": VOCAB_SIZE,
    "registered_special_tokens": len(SPECIAL_TOKENS),
    "document_boundary_token": DOC_BOUNDARY_TOKEN,
    "byte_alphabet_size": BYTE_ALPHABET_SIZE,
    "non_special_vocab_slots": VOCAB_SIZE - len(SPECIAL_TOKENS),
    "max_merge_created_slots_if_full": (
        VOCAB_SIZE - len(SPECIAL_TOKENS) - BYTE_ALPHABET_SIZE
    ),
})

vocab_contract


,0
total_vocab_size,16384
registered_special_tokens,1
document_boundary_token,<|endoftext|>
byte_alphabet_size,256
non_special_vocab_slots,16383
max_merge_created_slots_if_full,16127


## 7. Prove the boundary-token string does not occur naturally

A registered special token must have an unambiguous structural meaning. If the literal string `<|endoftext|>` already appeared inside an article, that natural text could be confused with the boundary marker once the tokenizer treats the string as a special symbol.

We therefore search only the development-visible reconstructed `train` and `validation` articles. The test split remains uninspected.

In [9]:
boundary_collision_counts = {}

for split_name in AUDIT_SPLITS:
    literal_matches = sum(
        DOC_BOUNDARY_TOKEN in article["text"]
        for article in articles_by_split[split_name]
    )
    boundary_collision_counts[split_name] = literal_matches
    assert literal_matches == 0, (
        f"{DOC_BOUNDARY_TOKEN!r} occurs literally in {split_name}: "
        f"{literal_matches} articles"
    )

print("✓ Boundary-token literal collision check passed.")
pd.Series(boundary_collision_counts, name="articles_with_literal_boundary_token")


✓ Boundary-token literal collision check passed.


,articles_with_literal_boundary_token
train,0
validation,0


## 8. Keep literal `<unk>` as ordinary corpus text

WikiText can contain the literal characters `<unk>` as part of the released corpus. We **do not** register `<unk>` as a tokenizer special token. Doing so would change the meaning of those existing corpus occurrences from ordinary text into a control symbol.

Instead, we simply measure how often the literal string appears in development-visible articles. This is an observation, not an assertion: the observed counts are recorded after execution.

In [10]:
LITERAL_UNK = "<unk>"

unk_inventory = pd.DataFrame(
    [
        {
            "split": split_name,
            "articles_with_literal_unk": sum(
                LITERAL_UNK in article["text"]
                for article in articles_by_split[split_name]
            ),
            "literal_unk_occurrences": sum(
                article["text"].count(LITERAL_UNK)
                for article in articles_by_split[split_name]
            ),
        }
        for split_name in AUDIT_SPLITS
    ]
).set_index("split")

unk_inventory


,articles_with_literal_unk,literal_unk_occurrences
split,,
train,0,0
validation,0,0


### Pause here

This chunk locks the special-token contract and checks the corpus against it. **Do not train the tokenizer yet.**

Execution gate for the next chunk:
- the boundary-token collision count must be zero for both train and validation;
- the observed literal `<unk>` counts should be recorded without turning `<unk>` into a special token; and
- the 16,384-token vocabulary accounting must remain explicit.

**Next reviewed chunk:** configure the actual byte-level BPE tokenizer and verify the special-token ID / byte-level mechanics before full tokenizer training.

# Chunk 3 — Probe the byte-level BPE mechanics

Before training on the full ~519M-character WikiText training corpus, we will build a **tiny probe tokenizer** using the structural settings already locked for the project.

This probe is deliberately small. It answers four mechanical questions:

1. Does the registered `<|endoftext|>` token receive the expected ID?
2. Does the ByteLevel alphabet really provide all 256 byte symbols?
3. Can text containing punctuation, Unicode, emoji, and newlines round-trip without an unknown token?
4. Is the boundary/EOS token inserted only when we explicitly supply it?

The probe is **not** the project tokenizer. A tiny corpus cannot learn 16,384 useful vocabulary entries, and we are not yet locking the full-corpus merge-frequency policy. Full WikiText tokenizer training remains the next phase after these mechanics pass.


## 9. Pin the tokenizer implementation

Tokenizer behavior is part of reproducibility, so this notebook now installs and checks Hugging Face `tokenizers==0.23.1`.

The project already owns linguistic normalization in `src/data.py`. The tokenizer therefore does **not** add a second text normalizer here. ByteLevel's job is different: it maps raw UTF-8 bytes into a reversible visible alphabet and applies GPT-style pre-tokenization.


In [11]:
import tokenizers
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers

EXPECTED_TOKENIZERS_VERSION = "0.23.1"

assert tokenizers.__version__ == EXPECTED_TOKENIZERS_VERSION, (
    f"Unexpected tokenizers version: {tokenizers.__version__}"
)

print(f"✓ tokenizers version pinned: {tokenizers.__version__}")


✓ tokenizers version pinned: 0.23.1


## 10. Instantiate the byte-level BPE components

The intended tokenizer has three core pieces:

- `models.BPE()` — the vocabulary starts from symbols and learns frequent pair merges.
- `pre_tokenizers.ByteLevel(add_prefix_space=False, use_regex=True)` — exposes every UTF-8 byte through a 256-symbol reversible alphabet and uses GPT-style splitting behavior.
- `decoders.ByteLevel()` — reverses the byte-to-visible-character mapping during decode.

`add_prefix_space=False` is intentional: we preserve whether the original text actually began with a space rather than silently manufacturing one.

The trainer reserves `<|endoftext|>` first and explicitly seeds the complete 256-symbol ByteLevel alphabet. The probe leaves `min_frequency` at the library default because that is a **full-training policy choice**, not something needed to verify these mechanics.


In [12]:
def build_byte_bpe_components(show_progress=False):
    tokenizer_object = Tokenizer(models.BPE())
    tokenizer_object.pre_tokenizer = pre_tokenizers.ByteLevel(
        add_prefix_space=False,
        use_regex=True,
    )
    tokenizer_object.decoder = decoders.ByteLevel()

    trainer_object = trainers.BpeTrainer(
        vocab_size=VOCAB_SIZE,
        special_tokens=SPECIAL_TOKENS,
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
        show_progress=show_progress,
    )
    return tokenizer_object, trainer_object


byte_alphabet = pre_tokenizers.ByteLevel.alphabet()

assert len(byte_alphabet) == BYTE_ALPHABET_SIZE
assert len(set(byte_alphabet)) == BYTE_ALPHABET_SIZE

probe_tokenizer, probe_trainer = build_byte_bpe_components()

print(f"✓ ByteLevel alphabet contains {len(byte_alphabet)} unique symbols.")
print(f"✓ Requested vocabulary ceiling remains {VOCAB_SIZE:,} total IDs.")


✓ ByteLevel alphabet contains 256 unique symbols.
✓ Requested vocabulary ceiling remains 16,384 total IDs.


## 11. Train only a tiny mechanics probe

A BPE model has no usable vocabulary until a trainer observes some text. We therefore train on four short, fixed strings—not on WikiText—to make the configured tokenizer operational.

Because the corpus is tiny, the probe will stop far below 16,384 entries. That is expected. What matters here is **ordering and coverage**, especially the special-token ID and the seeded byte alphabet.


In [13]:
PROBE_TEXTS = [
    "Hello, tokenizer mechanics.",
    "Byte-level BPE keeps punctuation: 3.5%, 1,000, 12:30.",
    "Unicode stays reversible: café naïve résumé.",
    "Emoji and symbols stay reversible too: 🙂 — £ €.",
]

probe_tokenizer.train_from_iterator(
    PROBE_TEXTS,
    trainer=probe_trainer,
    length=len(PROBE_TEXTS),
)

probe_vocab = probe_tokenizer.get_vocab()
probe_vocab_size = len(probe_vocab)
boundary_token_id = probe_tokenizer.token_to_id(DOC_BOUNDARY_TOKEN)
byte_symbol_ids = {
    symbol: probe_tokenizer.token_to_id(symbol)
    for symbol in byte_alphabet
}

assert boundary_token_id is not None
assert boundary_token_id == 0
assert all(token_id is not None for token_id in byte_symbol_ids.values())
assert probe_vocab_size >= 1 + BYTE_ALPHABET_SIZE

print(f"Observed boundary token ID: {boundary_token_id}")
print(f"Byte symbols present: {sum(v is not None for v in byte_symbol_ids.values())}/256")
print(f"Probe vocabulary size: {probe_vocab_size:,}")
print(f"Requested full-training vocabulary size: {VOCAB_SIZE:,}")
print("✓ Special-token ordering and complete byte coverage verified.")


Observed boundary token ID: 0
Byte symbols present: 256/256
Probe vocabulary size: 368
Requested full-training vocabulary size: 16,384
✓ Special-token ordering and complete byte coverage verified.


## 12. Verify byte-level reversibility

Byte-level tokenization is useful because unseen Unicode characters do not require an `<unk>` token. UTF-8 first represents a character as one or more bytes; the ByteLevel alphabet guarantees that every possible byte value already has a symbol.

The exact number of tokens below is **not** a quality metric—the probe has learned only a few toy merges. The important check is that encode → decode reconstructs each input exactly and that normal text does not acquire a boundary token automatically.


In [14]:
ROUNDTRIP_SAMPLES = [
    "Hello, world!",
    "café naïve résumé",
    "🙂 — £ €",
    "line one\nline two",
    " leading space",
]

roundtrip_rows = []

for text in ROUNDTRIP_SAMPLES:
    encoding = probe_tokenizer.encode(text)
    decoded = probe_tokenizer.decode(
        encoding.ids,
        skip_special_tokens=False,
    )

    assert decoded == text
    assert boundary_token_id not in encoding.ids

    roundtrip_rows.append(
        {
            "text": repr(text),
            "utf8_bytes": len(text.encode("utf-8")),
            "probe_tokens": len(encoding.ids),
            "tokens": encoding.tokens,
            "round_trip_exact": decoded == text,
            "boundary_auto_inserted": boundary_token_id in encoding.ids,
        }
    )

print("✓ All byte-level round-trip checks passed.")
pd.DataFrame(roundtrip_rows)


✓ All byte-level round-trip checks passed.


,text,utf8_bytes,probe_tokens,tokens,round_trip_exact,boundary_auto_inserted
0,"'Hello, world!'",13,9,"[Hello, ,, Ġ, w, o, r, l, d, !]",True,False
1,'café naïve résumé',21,4,"[caf, Ã©, ĠnaÃ¯ve, ĠrÃ©sumÃ©]",True,False
2,'🙂 — £ €',15,7,"[ðŁ, ĻĤ, ĠâĢĶ, ĠÂ£, Ġâ, Ĥ, ¬]",True,False
3,'line one\nline two',17,17,"[l, i, n, e, Ġ, o, n, e, Ċ, l, i, n, e, Ġ, t, ...",True,False
4,' leading space',14,13,"[Ġ, l, e, a, d, i, n, g, Ġs, p, a, c, e]",True,False


## 13. Prove that EOS is structural, atomic, and explicit

Registering `<|endoftext|>` gives that literal string one reserved token ID. It does **not** create a post-processor that appends EOS to every encoded sequence.

So there are two distinct operations:

- encode ordinary document text → **no boundary token appears**;
- explicitly include/append `<|endoftext|>` at a document boundary → the tokenizer emits the reserved ID exactly once.

That distinction is what lets the later corpus-construction code control article boundaries precisely and count them inside the exact 20M-token budget.


In [15]:
plain_text = "Document body."
explicit_boundary_text = plain_text + DOC_BOUNDARY_TOKEN

plain_encoding = probe_tokenizer.encode(plain_text)
explicit_encoding = probe_tokenizer.encode(explicit_boundary_text)

assert boundary_token_id not in plain_encoding.ids
assert explicit_encoding.ids.count(boundary_token_id) == 1
assert explicit_encoding.tokens.count(DOC_BOUNDARY_TOKEN) == 1

special_behavior = pd.Series(
    {
        "boundary_token": DOC_BOUNDARY_TOKEN,
        "boundary_token_id": boundary_token_id,
        "plain_ids_contain_boundary": boundary_token_id in plain_encoding.ids,
        "explicit_boundary_count": explicit_encoding.ids.count(boundary_token_id),
        "decode_keep_special": probe_tokenizer.decode(
            explicit_encoding.ids,
            skip_special_tokens=False,
        ),
        "decode_skip_special": probe_tokenizer.decode(
            explicit_encoding.ids,
            skip_special_tokens=True,
        ),
    }
)

print("✓ Boundary token is explicit rather than automatically inserted.")
special_behavior


✓ Boundary token is explicit rather than automatically inserted.


,0
boundary_token,<|endoftext|>
boundary_token_id,0
plain_ids_contain_boundary,False
explicit_boundary_count,1
decode_keep_special,Document body.<|endoftext|>
decode_skip_special,Document body.


### Pause here

Chunk 3 is a mechanics gate. **Do not train the full WikiText tokenizer yet.**

Before the full run, the outputs should establish:

- `tokenizers == 0.23.1`;
- `<|endoftext|>` has observed token ID **0** under the intended trainer ordering;
- all **256/256** ByteLevel symbols are present;
- punctuation, Unicode, emoji, newlines, and leading spaces round-trip exactly;
- ordinary text receives **no automatic boundary token**; and
- an explicitly supplied `<|endoftext|>` is recognized atomically exactly once.

A probe vocabulary smaller than 16,384 is expected because four toy strings cannot supply enough distinct useful merges.

**Next reviewed chunk:** lock the remaining full-training BPE policy (especially merge-frequency handling), then train the actual tokenizer on the complete normalized training split.


# Chunk 4 — Train the production byte-level BPE tokenizer

The mechanics probe passed. We can now train the **actual project tokenizer** on the complete normalized WikiText-103 training corpus reconstructed earlier in this notebook.

Production configuration:

- algorithm: byte-level BPE;
- `tokenizers==0.23.1`;
- total vocabulary target: **16,384** IDs;
- sole registered special token: `<|endoftext|>`;
- complete 256-symbol ByteLevel initial alphabet;
- `ByteLevel(add_prefix_space=False, use_regex=True)`;
- `min_frequency=2`;
- training input: all **28,472** normalized training articles, in reconstruction order;
- validation and test text are not used for tokenizer training.

## D-056 — BPE merge-frequency stopping rule

`min_frequency=2` is a **stopping rule**. The trainer repeatedly selects the most frequent remaining pair. If the best remaining pair occurs fewer than 2 times, training stops.

A pair that exists has count at least 1, so **2 is the smallest threshold that changes behavior**.

With `ByteLevel(use_regex=True)`, text is first split with the GPT-2-style regular expression. The trainer aggregates counts of those pre-tokenized units across the corpus, then weights pair counts by each unit's occurrence count.

**Prediction before training:** the floor will be non-binding. If so, the tokenizer reaches all 16,384 IDs and produces:

`16,384 - 1 special - 256 byte symbols = 16,127 learned merges`

We will measure that rather than assume it.


## 14. Freeze the production BPE configuration

No `limit_alphabet` is needed. ByteLevel can emit only members of its 256-symbol byte alphabet, and we explicitly seed all 256 symbols. We still assert full byte-symbol coverage after training.


In [16]:
import hashlib
import json
import time

BPE_MIN_FREQUENCY = 2
EXPECTED_BASE_VOCAB = len(SPECIAL_TOKENS) + BYTE_ALPHABET_SIZE
EXPECTED_LEARNED_MERGES = VOCAB_SIZE - EXPECTED_BASE_VOCAB

assert EXPECTED_BASE_VOCAB == 257
assert EXPECTED_LEARNED_MERGES == 16_127

def build_production_tokenizer(show_progress=True):
    tokenizer_object = Tokenizer(models.BPE())
    tokenizer_object.pre_tokenizer = pre_tokenizers.ByteLevel(
        add_prefix_space=False,
        use_regex=True,
    )
    tokenizer_object.decoder = decoders.ByteLevel()

    trainer_object = trainers.BpeTrainer(
        vocab_size=VOCAB_SIZE,
        min_frequency=BPE_MIN_FREQUENCY,
        special_tokens=SPECIAL_TOKENS,
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
        show_progress=show_progress,
    )
    return tokenizer_object, trainer_object

production_config = pd.Series({
    "tokenizers_version": tokenizers.__version__,
    "vocab_size_target": VOCAB_SIZE,
    "min_frequency": BPE_MIN_FREQUENCY,
    "registered_special_tokens": len(SPECIAL_TOKENS),
    "byte_alphabet_size": BYTE_ALPHABET_SIZE,
    "expected_learned_merges_if_full": EXPECTED_LEARNED_MERGES,
    "training_articles": len(articles_by_split["train"]),
    "add_prefix_space": False,
    "use_regex": True,
    "limit_alphabet": None,
})

production_config


,0
tokenizers_version,0.23.1
vocab_size_target,16384
min_frequency,2
registered_special_tokens,1
byte_alphabet_size,256
expected_learned_merges_if_full,16127
training_articles,28472
add_prefix_space,False
use_regex,True
limit_alphabet,None


## 15. Train on all 28,472 normalized training articles

`train_from_iterator` receives one reconstructed article string at a time, in canonical reconstruction order. `length=28_472` supplies the iterator length for progress reporting.

Only `articles_by_split["train"]` is used. We also record wall-clock time.


In [17]:
train_articles = articles_by_split["train"]

assert len(train_articles) == EXPECTED_ARTICLE_COUNTS["train"] == 28_472

production_tokenizer, production_trainer = build_production_tokenizer(
    show_progress=True
)

training_started = time.perf_counter()

production_tokenizer.train_from_iterator(
    (article["text"] for article in train_articles),
    trainer=production_trainer,
    length=len(train_articles),
)

tokenizer_training_seconds = time.perf_counter() - training_started

print(
    "✓ Production tokenizer training completed in "
    f"{tokenizer_training_seconds:,.2f} seconds "
    f"({tokenizer_training_seconds / 60:,.2f} minutes)."
)


✓ Production tokenizer training completed in 275.02 seconds (4.58 minutes).


## 16. Turn D-056 into evidence

We require:

1. vocabulary size = **16,384**;
2. `<|endoftext|>` ID = **0**;
3. all **256** ByteLevel symbols present;
4. exactly **16,127** learned merges.

Python's public `BPE` API does not expose `get_merges()`, so the merge count is read from the serialized tokenizer model.


In [18]:
production_vocab = production_tokenizer.get_vocab()
production_boundary_id = production_tokenizer.token_to_id(
    DOC_BOUNDARY_TOKEN
)

production_byte_symbol_ids = {
    symbol: production_tokenizer.token_to_id(symbol)
    for symbol in byte_alphabet
}

serialized_tokenizer = production_tokenizer.to_str()
serialized_state = json.loads(serialized_tokenizer)
serialized_merges = serialized_state["model"]["merges"]

actual_vocab_size = len(production_vocab)
actual_merge_count = len(serialized_merges)
present_byte_symbols = sum(
    token_id is not None
    for token_id in production_byte_symbol_ids.values()
)

assert actual_vocab_size == VOCAB_SIZE, (
    f"Vocabulary stopped at {actual_vocab_size:,}; "
    f"expected {VOCAB_SIZE:,}."
)
assert production_boundary_id == 0
assert present_byte_symbols == BYTE_ALPHABET_SIZE
assert actual_merge_count == EXPECTED_LEARNED_MERGES, (
    f"Observed {actual_merge_count:,} merges; "
    f"expected {EXPECTED_LEARNED_MERGES:,}."
)

production_training_evidence = pd.Series({
    "tokenizers_version": tokenizers.__version__,
    "training_articles": len(train_articles),
    "training_seconds": round(tokenizer_training_seconds, 3),
    "training_minutes": round(tokenizer_training_seconds / 60, 3),
    "final_vocab_size": actual_vocab_size,
    "boundary_token_id": production_boundary_id,
    "byte_symbols_present": present_byte_symbols,
    "learned_merges": actual_merge_count,
    "min_frequency": BPE_MIN_FREQUENCY,
    "min_frequency_bound": actual_merge_count < EXPECTED_LEARNED_MERGES,
})

print("✓ Production tokenizer contract passed.")
print("✓ D-056 prediction confirmed: min_frequency=2 did not bind.")
production_training_evidence


✓ Production tokenizer contract passed.
✓ D-056 prediction confirmed: min_frequency=2 did not bind.


,0
tokenizers_version,0.23.1
training_articles,28472
training_seconds,275.024
training_minutes,4.584
final_vocab_size,16384
boundary_token_id,0
byte_symbols_present,256
learned_merges,16127
min_frequency,2
min_frequency_bound,False


## 17. Inspect the learned merge sequence

The serialized merge list is ordered by learned BPE rank. We inspect the first and last five merges as a sanity check.

The Python trainer does not expose the frequency of the final accepted merge, so we do not invent that number. Reaching all 16,127 merge slots is sufficient to prove that the floor of 2 did not stop training early.


In [19]:
merge_preview = pd.DataFrame({
    "rank": (
        list(range(5))
        + list(range(actual_merge_count - 5, actual_merge_count))
    ),
    "merge": serialized_merges[:5] + serialized_merges[-5:],
})

merge_preview


,rank,merge
0,0,"[Ġ, t]"
1,1,"[h, e]"
2,2,"[Ġ, a]"
3,3,"[i, n]"
4,4,"[Ġt, he]"
5,16122,"[Ġcur, ric]"
6,16123,"[Ġkilog, rams]"
7,16124,"[c, os]"
8,16125,"[Ġex, empl]"
9,16126,"[ĠSt, op]"


## 18. Compute an in-memory SHA-256

We hash the complete serialized tokenizer JSON immediately after training.

The pinned implementation serializes vocabulary entries in token-ID order and merge rules in merge-rank order, so a repeated byte-for-byte checksum is a meaningful empirical reproducibility test for the same corpus, configuration, and library version.


In [20]:
production_tokenizer_sha256 = hashlib.sha256(
    serialized_tokenizer.encode("utf-8")
).hexdigest()

checksum_evidence = pd.Series({
    "sha256": production_tokenizer_sha256,
    "serialized_bytes": len(serialized_tokenizer.encode("utf-8")),
    "tokenizers_version": tokenizers.__version__,
    "training_seconds": round(tokenizer_training_seconds, 3),
})

checksum_evidence


,0
sha256,6ec601a267cec7c843df47927f53c4dd108c85a1d05931...
serialized_bytes,486750
tokenizers_version,0.23.1
training_seconds,275.024


## 19. Repeat production training once to test determinism

This is a second full training run in the same Colab session.

The implementation:

- aggregates pre-tokenized word counts, including parallel counting;
- sorts the initial alphabet deterministically;
- breaks equal-frequency merge ties by the pair's token-ID tuple;
- serializes vocabulary by token ID and merges by rank.

Those properties make determinism plausible. The stronger project evidence is empirical: train twice with the same pinned input/configuration and require identical serialized SHA-256 checksums.

If the second full run is impractical in the available runtime, change `RUN_DETERMINISM_REPEAT` to `False` and record that the repeat test was deferred.


In [21]:
RUN_DETERMINISM_REPEAT = True

if RUN_DETERMINISM_REPEAT:
    repeat_tokenizer, repeat_trainer = build_production_tokenizer(
        show_progress=True
    )

    repeat_started = time.perf_counter()

    repeat_tokenizer.train_from_iterator(
        (article["text"] for article in train_articles),
        trainer=repeat_trainer,
        length=len(train_articles),
    )

    repeat_training_seconds = time.perf_counter() - repeat_started
    repeat_serialized = repeat_tokenizer.to_str()
    repeat_sha256 = hashlib.sha256(
        repeat_serialized.encode("utf-8")
    ).hexdigest()

    repeat_merge_count = len(
        json.loads(repeat_serialized)["model"]["merges"]
    )

    assert repeat_tokenizer.get_vocab_size() == VOCAB_SIZE
    assert repeat_tokenizer.token_to_id(DOC_BOUNDARY_TOKEN) == 0
    assert repeat_merge_count == EXPECTED_LEARNED_MERGES
    assert repeat_sha256 == production_tokenizer_sha256, (
        "Tokenizer SHA-256 mismatch across identical training runs."
    )

    determinism_evidence = pd.Series({
        "first_sha256": production_tokenizer_sha256,
        "repeat_sha256": repeat_sha256,
        "checksums_match": repeat_sha256 == production_tokenizer_sha256,
        "first_training_seconds": round(tokenizer_training_seconds, 3),
        "repeat_training_seconds": round(repeat_training_seconds, 3),
        "tokenizers_version": tokenizers.__version__,
    })

    print("✓ Determinism repeat passed: SHA-256 checksums match.")
    display(determinism_evidence)
else:
    print(
        "Determinism repeat deferred; "
        "production tokenizer remains valid."
    )


✓ Determinism repeat passed: SHA-256 checksums match.


,0
first_sha256,6ec601a267cec7c843df47927f53c4dd108c85a1d05931...
repeat_sha256,6ec601a267cec7c843df47927f53c4dd108c85a1d05931...
checksums_match,True
first_training_seconds,275.024
repeat_training_seconds,316.751
tokenizers_version,0.23.1


### Pause here

Chunk 4 is the **production tokenizer-training gate**.

Do not construct the 20M-token model-training corpus yet.

The run should establish:

- `tokenizers==0.23.1`;
- training input = **28,472 training articles only**;
- final vocabulary = **16,384**;
- `<|endoftext|>` ID = **0**;
- ByteLevel symbols present = **256/256**;
- learned merges = **16,127**;
- therefore `min_frequency=2` was **non-binding**;
- first-run wall-clock time;
- first-run serialized SHA-256;
- and, if the repeat cell runs, an identical second SHA-256.

After this passes, the canonical docs will be synchronized:

- backfill **D-055** with special-token evidence from Chunks 2–3;
- add **D-056** with the stopping-rule wording;
- replace the prediction with the observed merge count;
- record whether the repeated checksum matched.

**Next reviewed chunk:** production tokenizer validation and persistence metadata, then construction of the exact 20,000,000-token article-sampled corpus.


# Chunk 5 — Persist and validate the production tokenizer artifact

Chunk 4 proved that the production tokenizer reaches the full 16,384-token vocabulary, learns exactly 16,127 merges, preserves all 256 ByteLevel symbols, and is deterministic across two identical training runs.

This chunk turns that in-memory tokenizer into a **reusable project artifact**.

Goals:

1. save the tokenizer using Hugging Face Tokenizers' unified JSON format;
2. compute the canonical SHA-256 over the **actual saved file bytes**;
3. reload the tokenizer from disk;
4. prove the reloaded tokenizer preserves the locked structural contract;
5. verify encode/decode equivalence on fixed samples and development-visible validation text;
6. record artifact metadata;
7. package the tokenizer and metadata for transfer back into the repository.

The test split remains untouched.


## 20. Save the tokenizer artifact

`Tokenizer.save(path, pretty=False)` writes the complete tokenizer state as one compact JSON file.

Using `pretty=False` intentionally matches the compact serialization used by `Tokenizer.to_str(pretty=False)`. That lets us verify that the bytes written to disk are exactly the serialization we already validated in memory.


In [25]:
from pathlib import Path
import hashlib
import json
import shutil

TOKENIZER_ARTIFACT_DIR = Path("/content/tokenizer_artifact")
TOKENIZER_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

TOKENIZER_JSON_PATH = TOKENIZER_ARTIFACT_DIR / "tokenizer.json"
TOKENIZER_METADATA_PATH = TOKENIZER_ARTIFACT_DIR / "tokenizer_metadata.json"
TOKENIZER_ZIP_BASE = Path("/content/tokenizer_artifact_bundle")

production_tokenizer.save(
    str(TOKENIZER_JSON_PATH),
    pretty=False,
)

assert TOKENIZER_JSON_PATH.exists()
assert TOKENIZER_JSON_PATH.stat().st_size > 0

print(f"✓ Saved production tokenizer: {TOKENIZER_JSON_PATH}")
print(f"Saved bytes: {TOKENIZER_JSON_PATH.stat().st_size:,}")


✓ Saved production tokenizer: /content/tokenizer_artifact/tokenizer.json
Saved bytes: 486,750


## 21. Compute the canonical SHA-256 from saved bytes

The checksum recorded from this point forward is the digest of the **persisted `tokenizer.json` bytes**, not a truncated DataFrame rendering and not merely an in-memory object fingerprint.

We also compare the saved bytes with `production_tokenizer.to_str(pretty=False)` to prove that persistence did not alter the serialized tokenizer.


In [26]:
tokenizer_file_bytes = TOKENIZER_JSON_PATH.read_bytes()
canonical_tokenizer_sha256 = hashlib.sha256(
    tokenizer_file_bytes
).hexdigest()

compact_in_memory_bytes = production_tokenizer.to_str(
    pretty=False
).encode("utf-8")
in_memory_sha256 = hashlib.sha256(
    compact_in_memory_bytes
).hexdigest()

assert tokenizer_file_bytes == compact_in_memory_bytes
assert canonical_tokenizer_sha256 == in_memory_sha256
assert len(canonical_tokenizer_sha256) == 64

print("✓ Persisted bytes exactly match compact in-memory serialization.")
print(f"Canonical tokenizer SHA-256: {canonical_tokenizer_sha256}")
print(f"Tokenizer file size: {len(tokenizer_file_bytes):,} bytes")


✓ Persisted bytes exactly match compact in-memory serialization.
Canonical tokenizer SHA-256: 6ec601a267cec7c843df47927f53c4dd108c85a1d059318aeec4442c7274604f
Tokenizer file size: 486,750 bytes


## 22. Reload from disk and re-check the structural contract

A saved tokenizer is useful only if a fresh process can reload it and recover the same behavior.

The reloaded artifact must preserve:

- total vocabulary = **16,384**;
- `<|endoftext|>` ID = **0**;
- all **256** ByteLevel symbols;
- exactly **16,127** BPE merges;
- no padding configuration;
- no truncation configuration;
- no automatic post-processor that inserts special tokens.


In [27]:
reloaded_tokenizer = Tokenizer.from_file(
    str(TOKENIZER_JSON_PATH)
)

reloaded_state = json.loads(
    reloaded_tokenizer.to_str(pretty=False)
)
reloaded_merges = reloaded_state["model"]["merges"]

reloaded_byte_symbol_ids = {
    symbol: reloaded_tokenizer.token_to_id(symbol)
    for symbol in byte_alphabet
}

assert reloaded_tokenizer.get_vocab_size() == VOCAB_SIZE
assert (
    reloaded_tokenizer.token_to_id(DOC_BOUNDARY_TOKEN)
    == production_boundary_id
    == 0
)
assert all(
    token_id is not None
    for token_id in reloaded_byte_symbol_ids.values()
)
assert len(reloaded_merges) == EXPECTED_LEARNED_MERGES
assert reloaded_tokenizer.padding is None
assert reloaded_tokenizer.truncation is None
assert reloaded_tokenizer.post_processor is None

print("✓ Reloaded tokenizer preserves the complete structural contract.")
pd.Series({
    "vocab_size": reloaded_tokenizer.get_vocab_size(),
    "boundary_token_id": reloaded_tokenizer.token_to_id(
        DOC_BOUNDARY_TOKEN
    ),
    "byte_symbols_present": sum(
        token_id is not None
        for token_id in reloaded_byte_symbol_ids.values()
    ),
    "learned_merges": len(reloaded_merges),
    "padding_configured": reloaded_tokenizer.padding is not None,
    "truncation_configured": reloaded_tokenizer.truncation is not None,
    "post_processor_configured": (
        reloaded_tokenizer.post_processor is not None
    ),
})


✓ Reloaded tokenizer preserves the complete structural contract.


,0
vocab_size,16384
boundary_token_id,0
byte_symbols_present,256
learned_merges,16127
padding_configured,False
truncation_configured,False
post_processor_configured,False


## 23. Verify encode/decode equivalence after reload

We compare the original in-memory production tokenizer with the reloaded artifact on:

- the same fixed Unicode/punctuation samples used earlier;
- explicit boundary-token behavior;
- deterministic snippets from the **validation** articles.

For every sample, the two tokenizers must produce identical IDs and identical decoded text.

Validation text is development-visible and is used only for checking tokenizer behavior. It was not used to train the tokenizer. The test split remains untouched.


In [28]:
artifact_validation_samples = list(ROUNDTRIP_SAMPLES)

for article in articles_by_split["validation"][:5]:
    # Keep the check compact while still using real held-out prose.
    artifact_validation_samples.append(article["text"][:1000])

equivalence_rows = []

for text in artifact_validation_samples:
    original_encoding = production_tokenizer.encode(text)
    reloaded_encoding = reloaded_tokenizer.encode(text)

    original_decoded = production_tokenizer.decode(
        original_encoding.ids,
        skip_special_tokens=False,
    )
    reloaded_decoded = reloaded_tokenizer.decode(
        reloaded_encoding.ids,
        skip_special_tokens=False,
    )

    assert original_encoding.ids == reloaded_encoding.ids
    assert original_encoding.tokens == reloaded_encoding.tokens
    assert original_decoded == text
    assert reloaded_decoded == text
    assert boundary_token_id not in original_encoding.ids
    assert boundary_token_id not in reloaded_encoding.ids

    equivalence_rows.append({
        "characters": len(text),
        "utf8_bytes": len(text.encode("utf-8")),
        "tokens": len(reloaded_encoding.ids),
        "ids_match": original_encoding.ids == reloaded_encoding.ids,
        "round_trip_exact": reloaded_decoded == text,
        "boundary_auto_inserted": (
            boundary_token_id in reloaded_encoding.ids
        ),
    })

explicit_reloaded = reloaded_tokenizer.encode(
    "Document body." + DOC_BOUNDARY_TOKEN
)
assert explicit_reloaded.ids.count(boundary_token_id) == 1
assert explicit_reloaded.tokens.count(DOC_BOUNDARY_TOKEN) == 1

print("✓ Reloaded tokenizer matches the production tokenizer exactly.")
print("✓ Ordinary encoding still does not auto-insert the boundary token.")
print("✓ Explicit boundary token is still recognized atomically once.")
pd.DataFrame(equivalence_rows)


✓ Reloaded tokenizer matches the production tokenizer exactly.
✓ Ordinary encoding still does not auto-insert the boundary token.
✓ Explicit boundary token is still recognized atomically once.


,characters,utf8_bytes,tokens,ids_match,round_trip_exact,boundary_auto_inserted
0,13,13,5,True,True,False
1,17,21,12,True,True,False
2,7,15,7,True,True,False
3,17,17,5,True,True,False
4,14,14,2,True,True,False
5,1000,1012,282,True,True,False
6,1000,1006,232,True,True,False
7,1000,1010,252,True,True,False
8,1000,1012,270,True,True,False
9,1000,1000,239,True,True,False


## 24. Measure tokenizer efficiency on the complete validation article set

This is not a language-model metric. It is a tokenizer-efficiency measurement.

We encode all 60 reconstructed validation articles **without appending document-boundary tokens** and report:

- normalized characters;
- UTF-8 bytes;
- tokenizer-produced lexical tokens;
- characters per token;
- bytes per token.

Why bytes/token is especially useful here: byte-level BPE begins from bytes, so it provides a direct compression view that remains meaningful for Unicode text.

The validation split is appropriate for this diagnostic because it was excluded from tokenizer training. Test remains untouched.


In [29]:
validation_texts = [
    article["text"]
    for article in articles_by_split["validation"]
]

validation_characters = sum(
    len(text)
    for text in validation_texts
)
validation_utf8_bytes = sum(
    len(text.encode("utf-8"))
    for text in validation_texts
)
validation_token_count = sum(
    len(reloaded_tokenizer.encode(text).ids)
    for text in validation_texts
)

assert len(validation_texts) == EXPECTED_ARTICLE_COUNTS["validation"]
assert validation_token_count > 0

validation_tokenizer_efficiency = pd.Series({
    "validation_articles": len(validation_texts),
    "characters": validation_characters,
    "utf8_bytes": validation_utf8_bytes,
    "tokens_no_boundaries": validation_token_count,
    "characters_per_token": (
        validation_characters / validation_token_count
    ),
    "bytes_per_token": (
        validation_utf8_bytes / validation_token_count
    ),
})

validation_tokenizer_efficiency


,0
validation_articles,6.000000e+01
characters,1.102768e+06
utf8_bytes,1.104866e+06
tokens_no_boundaries,2.565790e+05
characters_per_token,4.297967e+00
bytes_per_token,4.306144e+00


## 25. Write tokenizer metadata

The metadata file records the decisions and evidence needed to identify this tokenizer without opening the notebook.

It includes:

- upstream dataset identity and immutable Hub revision;
- pinned preprocessing source revision;
- `tokenizers` library version;
- training article count;
- BPE configuration;
- vocabulary and merge counts;
- boundary-token contract;
- ByteLevel alphabet size;
- canonical persisted-file SHA-256;
- artifact byte size;
- first-run training time;
- determinism-repeat result;
- validation tokenizer-efficiency metrics.

The exact 20M-token corpus manifest will later reference this tokenizer checksum.


In [30]:
determinism_repeat_match = None
repeat_seconds_for_metadata = None

if "determinism_evidence" in globals():
    determinism_repeat_match = bool(
        determinism_evidence["checksums_match"]
    )
    repeat_seconds_for_metadata = float(
        determinism_evidence["repeat_training_seconds"]
    )

tokenizer_metadata = {
    "dataset": DATASET_ID,
    "dataset_config": DATASET_CONFIG,
    "dataset_revision": DATASET_REVISION,
    "data_pipeline_revision": DATA_PIPELINE_REVISION,
    "tokenizers_version": tokenizers.__version__,
    "training_split": "train",
    "training_articles": len(train_articles),
    "vocab_size": VOCAB_SIZE,
    "special_tokens": SPECIAL_TOKENS,
    "document_boundary_token": DOC_BOUNDARY_TOKEN,
    "document_boundary_token_id": production_boundary_id,
    "byte_alphabet_size": BYTE_ALPHABET_SIZE,
    "learned_merges": actual_merge_count,
    "min_frequency": BPE_MIN_FREQUENCY,
    "add_prefix_space": False,
    "use_regex": True,
    "padding": None,
    "bos_token": None,
    "unk_token": None,
    "automatic_boundary_insertion": False,
    "tokenizer_sha256": canonical_tokenizer_sha256,
    "tokenizer_bytes": len(tokenizer_file_bytes),
    "first_training_seconds": round(
        float(tokenizer_training_seconds), 3
    ),
    "determinism_repeat_checksums_match": (
        determinism_repeat_match
    ),
    "repeat_training_seconds": repeat_seconds_for_metadata,
    "validation_articles": len(validation_texts),
    "validation_tokens_no_boundaries": validation_token_count,
    "validation_characters_per_token": float(
        validation_characters / validation_token_count
    ),
    "validation_bytes_per_token": float(
        validation_utf8_bytes / validation_token_count
    ),
}

TOKENIZER_METADATA_PATH.write_text(
    json.dumps(
        tokenizer_metadata,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
    ) + "\n",
    encoding="utf-8",
)

metadata_sha256 = hashlib.sha256(
    TOKENIZER_METADATA_PATH.read_bytes()
).hexdigest()

print(f"✓ Wrote metadata: {TOKENIZER_METADATA_PATH}")
print(f"Tokenizer SHA-256: {canonical_tokenizer_sha256}")
print(f"Metadata SHA-256:  {metadata_sha256}")


✓ Wrote metadata: /content/tokenizer_artifact/tokenizer_metadata.json
Tokenizer SHA-256: 6ec601a267cec7c843df47927f53c4dd108c85a1d059318aeec4442c7274604f
Metadata SHA-256:  94c08fdb237c3526cb91c9838e27fbf3b5613ccc434132f658c710683ab328da


## 26. Package the artifact for repository transfer

The ZIP is only a transfer convenience. The repository should ultimately store the two files uncompressed under:

```text
results/tokenizer/tokenizer.json
results/tokenizer/tokenizer_metadata.json
```

The canonical tokenizer checksum is the SHA-256 of `tokenizer.json` itself, not the ZIP.


In [31]:
archive_path = shutil.make_archive(
    str(TOKENIZER_ZIP_BASE),
    "zip",
    root_dir=TOKENIZER_ARTIFACT_DIR,
)

archive_path = Path(archive_path)
archive_sha256 = hashlib.sha256(
    archive_path.read_bytes()
).hexdigest()

print(f"✓ Created artifact bundle: {archive_path}")
print(f"Bundle size: {archive_path.stat().st_size:,} bytes")
print(f"Bundle SHA-256: {archive_sha256}")
print()
print("Repository target:")
print("  results/tokenizer/tokenizer.json")
print("  results/tokenizer/tokenizer_metadata.json")


✓ Created artifact bundle: /content/tokenizer_artifact_bundle.zip
Bundle size: 174,136 bytes
Bundle SHA-256: 404a8620135bbd6dd47d17dde22aeb355854b8db5ad76ec4d82bdab302912f75

Repository target:
  results/tokenizer/tokenizer.json
  results/tokenizer/tokenizer_metadata.json


## 27. Download the artifact bundle

Running this cell in Colab downloads one ZIP containing both production tokenizer files.

Attach that ZIP together with the executed notebook when you return here.


In [32]:
from google.colab import files

files.download(str(archive_path))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Pause here

Chunk 5 is the **persisted-tokenizer artifact gate**.

Do not construct the 20M-token training corpus yet.

Before proceeding, we should have observed:

- saved `tokenizer.json`;
- a printed full **64-character canonical SHA-256** of the saved bytes;
- saved bytes exactly equal compact in-memory serialization;
- successful reload with:
  - vocab = **16,384**,
  - boundary ID = **0**,
  - byte symbols = **256/256**,
  - merges = **16,127**,
  - no padding,
  - no truncation,
  - no post-processor;
- exact encode/decode equivalence after reload;
- no automatic boundary insertion;
- explicit boundary recognized atomically once;
- tokenizer-efficiency metrics over all **60 validation articles**;
- `tokenizer_metadata.json`;
- downloaded `tokenizer_artifact_bundle.zip`.

**Next reviewed chunk:** deterministic article permutation and construction of the exact 20,000,000-token model-training corpus and manifest.


# Chunk 6 — Construct the exact 20,000,000-token training corpus

The tokenizer is now a persisted, reloadable artifact. We can finally construct the **controlled model-training corpus**.

The corpus-selection contract was established earlier:

1. use only the verified WikiText-103 **training** articles;
2. create `np.random.default_rng(42)` immediately before article permutation;
3. permute the 28,472 verified training-article indices once;
4. encode articles in that deterministic order;
5. append one `<|endoftext|>` boundary token after each complete article;
6. count boundary tokens inside the exact 20,000,000-token budget;
7. stop at exactly 20,000,000 tokens, truncating only the final selected article sequence if necessary;
8. if the budget ends before the final article's boundary token, that boundary is absent;
9. save an ordered manifest sufficient to reconstruct the exact stream.

The test split remains untouched.

## Storage policy

The notebook will materialize the exact token stream as little-endian `uint16` because the vocabulary has only 16,384 IDs. That produces exactly **40,000,000 bytes**.

The binary stream is **generated evidence**, not the canonical source artifact. We will commit:

- `corpus_summary.json`
- `corpus_manifest.jsonl`

and record the SHA-256 of the generated 40 MB token stream.

The binary itself does not need to live in Git because it can be regenerated exactly from:

**pinned dataset + pinned preprocessing + tokenizer checksum + seed + ordered manifest**

The notebook will prove that by reconstructing the stream from the manifest and requiring the same SHA-256.


## 28. Freeze the corpus-construction constants

The tokenizer artifact checksum is asserted before corpus construction begins. This prevents an accidentally different tokenizer from generating a superficially similar 20M-token corpus.


In [33]:
import numpy as np

CORPUS_TOKEN_BUDGET = 20_000_000
CORPUS_SAMPLING_SEED = 42
TOKEN_DTYPE = np.dtype("<u2")
EXPECTED_TOKENIZER_SHA256 = (
    "6ec601a267cec7c843df47927f53c4dd"
    "108c85a1d059318aeec4442c7274604f"
)
LOCKED_CONTEXT_LENGTH = 512

assert canonical_tokenizer_sha256 == EXPECTED_TOKENIZER_SHA256
assert reloaded_tokenizer.get_vocab_size() <= np.iinfo(TOKEN_DTYPE).max + 1
assert len(train_articles) == EXPECTED_ARTICLE_COUNTS["train"] == 28_472
assert CORPUS_TOKEN_BUDGET % LOCKED_CONTEXT_LENGTH == 256

corpus_config = pd.Series({
    "token_budget": CORPUS_TOKEN_BUDGET,
    "sampling_seed": CORPUS_SAMPLING_SEED,
    "token_dtype": TOKEN_DTYPE.str,
    "tokenizer_sha256": canonical_tokenizer_sha256,
    "train_articles_available": len(train_articles),
    "context_length": LOCKED_CONTEXT_LENGTH,
    "tail_if_naively_partitioned_at_512": (
        CORPUS_TOKEN_BUDGET % LOCKED_CONTEXT_LENGTH
    ),
})

corpus_config


,0
token_budget,20000000
sampling_seed,42
token_dtype,<u2
tokenizer_sha256,6ec601a267cec7c843df47927f53c4dd108c85a1d05931...
train_articles_available,28472
context_length,512
tail_if_naively_partitioned_at_512,256


## 29. Create the deterministic article permutation

The RNG is instantiated **immediately before** the permutation, exactly as locked in the sampling contract.

We also hash the complete permutation as little-endian unsigned 32-bit integers. This is an additional reproducibility fingerprint for the article ordering.


In [34]:
rng = np.random.default_rng(CORPUS_SAMPLING_SEED)
article_permutation = rng.permutation(len(train_articles))

article_permutation_u32 = np.asarray(
    article_permutation,
    dtype=np.dtype("<u4"),
)
permutation_sha256 = hashlib.sha256(
    article_permutation_u32.tobytes(order="C")
).hexdigest()

# Cheap independent repeat of the permutation itself.
repeat_rng = np.random.default_rng(CORPUS_SAMPLING_SEED)
repeat_permutation = repeat_rng.permutation(len(train_articles))

assert np.array_equal(article_permutation, repeat_permutation)
assert sorted(article_permutation.tolist()) == list(range(len(train_articles)))

print("✓ Deterministic article permutation reproduced exactly.")
print(f"Permutation SHA-256: {permutation_sha256}")
print("First 10 permuted article indices:")
print(article_permutation[:10].tolist())


✓ Deterministic article permutation reproduced exactly.
Permutation SHA-256: d4e368c0c22c1ea044133f7648466201450e66dc170da8ba67235fc1cd3b836c
First 10 permuted article indices:
[25736, 8546, 16806, 24733, 24260, 9549, 1206, 12654, 8387, 6816]


## 30. Materialize exactly 20,000,000 tokens

The token stream is preallocated as little-endian `uint16`, requiring only 40 MB.

For each selected article we record:

- selection rank;
- original reconstructed article index;
- stable article ID;
- full text-token count;
- full sequence-token count including the boundary;
- included text-token count;
- whether the boundary was included;
- whether text itself was truncated;
- whether the article sequence was truncated before completion;
- start and end offsets in the final corpus.

No token from validation or test enters this stream.


In [35]:
corpus_tokens = np.empty(
    CORPUS_TOKEN_BUDGET,
    dtype=TOKEN_DTYPE,
)

corpus_manifest_rows = []
cursor = 0

CORPUS_BOUNDARY_ID = reloaded_tokenizer.token_to_id(
    DOC_BOUNDARY_TOKEN
)
assert CORPUS_BOUNDARY_ID == 0

for selection_rank, article_index in enumerate(article_permutation):
    if cursor == CORPUS_TOKEN_BUDGET:
        break

    article = train_articles[int(article_index)]
    text_ids = reloaded_tokenizer.encode(article["text"]).ids

    # Literal boundary collisions were already audited as zero, and ordinary
    # encoding must not manufacture the special boundary ID.
    assert CORPUS_BOUNDARY_ID not in text_ids

    full_text_tokens = len(text_ids)
    full_sequence_tokens = full_text_tokens + 1
    remaining = CORPUS_TOKEN_BUDGET - cursor
    corpus_start = cursor

    if full_sequence_tokens <= remaining:
        included_text_tokens = full_text_tokens
        boundary_included = True
    else:
        included_text_tokens = min(full_text_tokens, remaining)
        boundary_included = False

    if included_text_tokens:
        corpus_tokens[
            cursor : cursor + included_text_tokens
        ] = np.asarray(
            text_ids[:included_text_tokens],
            dtype=TOKEN_DTYPE,
        )
        cursor += included_text_tokens

    if boundary_included:
        corpus_tokens[cursor] = CORPUS_BOUNDARY_ID
        cursor += 1

    included_sequence_tokens = cursor - corpus_start
    text_truncated = included_text_tokens < full_text_tokens
    sequence_truncated = (
        included_sequence_tokens < full_sequence_tokens
    )

    corpus_manifest_rows.append({
        "selection_rank": int(selection_rank),
        "article_index": int(article_index),
        "article_id": article["article_id"],
        "full_text_tokens": int(full_text_tokens),
        "full_sequence_tokens": int(full_sequence_tokens),
        "included_text_tokens": int(included_text_tokens),
        "boundary_included": bool(boundary_included),
        "included_sequence_tokens": int(included_sequence_tokens),
        "text_truncated": bool(text_truncated),
        "sequence_truncated": bool(sequence_truncated),
        "corpus_start": int(corpus_start),
        "corpus_end_exclusive": int(cursor),
    })

assert cursor == CORPUS_TOKEN_BUDGET
assert len(corpus_manifest_rows) > 0
assert corpus_manifest_rows[0]["corpus_start"] == 0
assert corpus_manifest_rows[-1]["corpus_end_exclusive"] == CORPUS_TOKEN_BUDGET

# Every record except possibly the final one must be complete.
for row in corpus_manifest_rows[:-1]:
    assert row["sequence_truncated"] is False
    assert row["boundary_included"] is True
    assert (
        row["included_sequence_tokens"]
        == row["full_sequence_tokens"]
    )

# If the final sequence is truncated, its boundary cannot be included.
if corpus_manifest_rows[-1]["sequence_truncated"]:
    assert corpus_manifest_rows[-1]["boundary_included"] is False

print(f"✓ Constructed exactly {cursor:,} tokens.")
print(f"Selected article records: {len(corpus_manifest_rows):,}")
print(
    "Final selected article:",
    corpus_manifest_rows[-1]["article_id"],
)


✓ Constructed exactly 20,000,000 tokens.
Selected article records: 4,604
Final selected article: train:article-02505:row-156360


## 31. Audit boundary accounting and final truncation

Because ordinary article encoding cannot emit token ID 0, every ID 0 in the final corpus must correspond to an explicitly appended document boundary.

This gives us a strong accounting identity:

`number of ID-0 tokens in corpus == number of manifest rows with boundary_included=True`


In [36]:
selected_article_count = len(corpus_manifest_rows)
boundary_tokens_included = sum(
    row["boundary_included"]
    for row in corpus_manifest_rows
)
observed_boundary_ids = int(
    np.count_nonzero(corpus_tokens == CORPUS_BOUNDARY_ID)
)
included_text_tokens_total = sum(
    row["included_text_tokens"]
    for row in corpus_manifest_rows
)

assert observed_boundary_ids == boundary_tokens_included
assert (
    included_text_tokens_total + boundary_tokens_included
    == CORPUS_TOKEN_BUDGET
)
assert int(corpus_tokens.max()) < VOCAB_SIZE

final_row = corpus_manifest_rows[-1]

corpus_accounting = pd.Series({
    "total_tokens": CORPUS_TOKEN_BUDGET,
    "selected_article_records": selected_article_count,
    "included_text_tokens": included_text_tokens_total,
    "boundary_tokens_included": boundary_tokens_included,
    "observed_boundary_id_count": observed_boundary_ids,
    "final_article_id": final_row["article_id"],
    "final_full_text_tokens": final_row["full_text_tokens"],
    "final_included_text_tokens": final_row["included_text_tokens"],
    "final_boundary_included": final_row["boundary_included"],
    "final_text_truncated": final_row["text_truncated"],
    "final_sequence_truncated": final_row["sequence_truncated"],
})

print("✓ Boundary accounting is exact.")
corpus_accounting


✓ Boundary accounting is exact.


,0
total_tokens,20000000
selected_article_records,4604
included_text_tokens,19995397
boundary_tokens_included,4603
observed_boundary_id_count,4603
final_article_id,train:article-02505:row-156360
final_full_text_tokens,4410
final_included_text_tokens,1312
final_boundary_included,False
final_text_truncated,True


## 32. Inspect the beginning and end of the ordered manifest

We inspect metadata only—not article text—to make the selection auditable without dumping corpus content into the notebook.


In [37]:
manifest_preview = pd.concat(
    [
        pd.DataFrame(corpus_manifest_rows[:5]),
        pd.DataFrame(corpus_manifest_rows[-5:]),
    ],
    ignore_index=True,
)

manifest_preview


,selection_rank,article_index,article_id,full_text_tokens,full_sequence_tokens,included_text_tokens,boundary_included,included_sequence_tokens,text_truncated,sequence_truncated,corpus_start,corpus_end_exclusive
0,0,25736,train:article-25736:row-1624198,2189,2190,2189,True,2190,False,False,0,2190
1,1,8546,train:article-08546:row-538354,5957,5958,5957,True,5958,False,False,2190,8148
2,2,16806,train:article-16806:row-1056052,3388,3389,3388,True,3389,False,False,8148,11537
3,3,24733,train:article-24733:row-1559384,2437,2438,2437,True,2438,False,False,11537,13975
4,4,24260,train:article-24260:row-1529683,2583,2584,2583,True,2584,False,False,13975,16559
5,4599,4949,train:article-04949:row-309822,2269,2270,2269,True,2270,False,False,19990379,19992649
6,4600,21909,train:article-21909:row-1381824,2687,2688,2687,True,2688,False,False,19992649,19995337
7,4601,16560,train:article-16560:row-1040683,2092,2093,2092,True,2093,False,False,19995337,19997430
8,4602,6101,train:article-06101:row-383361,1257,1258,1257,True,1258,False,False,19997430,19998688
9,4603,2505,train:article-02505:row-156360,4410,4411,1312,False,1312,True,True,19998688,20000000


## 33. Hash the exact token stream

The canonical corpus fingerprint is SHA-256 over the raw **little-endian uint16 token-ID bytes** in corpus order.

This definition is deliberately independent of NumPy `.npy` headers or ZIP/container metadata.

For 20,000,000 `uint16` IDs the raw stream must be exactly **40,000,000 bytes**.


In [38]:
CORPUS_BINARY_PATH = Path("/content/train_tokens.uint16.bin")

raw_corpus_bytes = corpus_tokens.tobytes(order="C")
corpus_token_sha256 = hashlib.sha256(
    raw_corpus_bytes
).hexdigest()

assert len(raw_corpus_bytes) == CORPUS_TOKEN_BUDGET * 2 == 40_000_000

CORPUS_BINARY_PATH.write_bytes(raw_corpus_bytes)
assert CORPUS_BINARY_PATH.stat().st_size == 40_000_000

print(f"✓ Raw token stream bytes: {len(raw_corpus_bytes):,}")
print(f"Corpus token-stream SHA-256: {corpus_token_sha256}")
print(
    "Binary materialized locally but will not be committed to Git:",
    CORPUS_BINARY_PATH,
)


✓ Raw token stream bytes: 40,000,000
Corpus token-stream SHA-256: 4101d5b18c38558a58110f54a161763186ab5318111366486ebbfa0a3fe584fa
Binary materialized locally but will not be committed to Git: /content/train_tokens.uint16.bin


## 34. Write the ordered manifest and corpus summary

`corpus_manifest.jsonl` contains one JSON object per selected article record, in the exact order consumed by the corpus.

`corpus_summary.json` records the provenance and aggregate evidence required to identify the corpus.

The guarded Hugging Face `_fingerprint` values are included only as **local cache-state diagnostics**. The immutable Hub revision remains the upstream dataset identity.


In [39]:
CORPUS_ARTIFACT_DIR = Path("/content/corpus_artifact")
CORPUS_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

CORPUS_MANIFEST_PATH = (
    CORPUS_ARTIFACT_DIR / "corpus_manifest.jsonl"
)
CORPUS_SUMMARY_PATH = (
    CORPUS_ARTIFACT_DIR / "corpus_summary.json"
)

manifest_text = "".join(
    json.dumps(row, sort_keys=True, ensure_ascii=False) + "\n"
    for row in corpus_manifest_rows
)
CORPUS_MANIFEST_PATH.write_text(
    manifest_text,
    encoding="utf-8",
)

manifest_sha256 = hashlib.sha256(
    CORPUS_MANIFEST_PATH.read_bytes()
).hexdigest()

local_dataset_fingerprints = {
    split_name: getattr(raw_dataset[split_name], "_fingerprint", None)
    for split_name in raw_dataset
}

corpus_summary = {
    "dataset": DATASET_ID,
    "dataset_config": DATASET_CONFIG,
    "dataset_revision": DATASET_REVISION,
    "local_dataset_fingerprints_diagnostic": (
        local_dataset_fingerprints
    ),
    "data_pipeline_revision": DATA_PIPELINE_REVISION,
    "tokenizer_sha256": canonical_tokenizer_sha256,
    "tokenizers_version": tokenizers.__version__,
    "sampling_split": "train",
    "sampling_seed": CORPUS_SAMPLING_SEED,
    "available_train_articles": len(train_articles),
    "article_permutation_sha256": permutation_sha256,
    "selected_article_records": selected_article_count,
    "token_budget": CORPUS_TOKEN_BUDGET,
    "token_dtype": TOKEN_DTYPE.str,
    "raw_token_bytes": len(raw_corpus_bytes),
    "token_stream_sha256": corpus_token_sha256,
    "included_text_tokens": included_text_tokens_total,
    "boundary_token": DOC_BOUNDARY_TOKEN,
    "boundary_token_id": CORPUS_BOUNDARY_ID,
    "boundary_tokens_included": boundary_tokens_included,
    "final_article_id": final_row["article_id"],
    "final_article_index": final_row["article_index"],
    "final_full_text_tokens": final_row["full_text_tokens"],
    "final_included_text_tokens": (
        final_row["included_text_tokens"]
    ),
    "final_boundary_included": (
        final_row["boundary_included"]
    ),
    "final_text_truncated": final_row["text_truncated"],
    "final_sequence_truncated": (
        final_row["sequence_truncated"]
    ),
    "manifest_sha256": manifest_sha256,
    "context_length_locked_for_later_training": (
        LOCKED_CONTEXT_LENGTH
    ),
    "unpacked_tail_mod_context_length": (
        CORPUS_TOKEN_BUDGET % LOCKED_CONTEXT_LENGTH
    ),
    "test_text_inspected_or_encoded": False,
}

CORPUS_SUMMARY_PATH.write_text(
    json.dumps(
        corpus_summary,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
    ) + "\n",
    encoding="utf-8",
)

summary_sha256 = hashlib.sha256(
    CORPUS_SUMMARY_PATH.read_bytes()
).hexdigest()

print(f"✓ Wrote manifest: {CORPUS_MANIFEST_PATH}")
print(f"Manifest SHA-256: {manifest_sha256}")
print(f"✓ Wrote summary: {CORPUS_SUMMARY_PATH}")
print(f"Summary SHA-256:  {summary_sha256}")


✓ Wrote manifest: /content/corpus_artifact/corpus_manifest.jsonl
Manifest SHA-256: 4a00196b39311a6c2e2790780e8fc43316f24a014d3d3649028b10a671f8d3fe
✓ Wrote summary: /content/corpus_artifact/corpus_summary.json
Summary SHA-256:  7b4c697d430c28ed9581ef1c5a65a12779d06d2828f33e222a3d1c36eacd9224


## 35. Reconstruct the corpus from the manifest and verify its hash

This is the strongest test in the chunk.

We discard reliance on the original in-memory token array and independently walk the saved manifest. For each manifest row we:

1. locate the article by its recorded reconstruction index;
2. assert its stable article ID;
3. re-encode it with the persisted tokenizer;
4. include exactly the recorded number of text tokens;
5. append the boundary only when the manifest says it was included;
6. stream the resulting little-endian `uint16` bytes into a new SHA-256 hash.

The reconstructed stream must contain exactly 20,000,000 tokens and must reproduce the canonical token-stream checksum.


In [40]:
reconstruction_hasher = hashlib.sha256()
reconstructed_token_count = 0

saved_manifest_rows = [
    json.loads(line)
    for line in CORPUS_MANIFEST_PATH.read_text(
        encoding="utf-8"
    ).splitlines()
    if line
]

assert saved_manifest_rows == corpus_manifest_rows

boundary_bytes = np.asarray(
    [CORPUS_BOUNDARY_ID],
    dtype=TOKEN_DTYPE,
).tobytes(order="C")

for row in saved_manifest_rows:
    article = train_articles[row["article_index"]]
    assert article["article_id"] == row["article_id"]

    text_ids = reloaded_tokenizer.encode(
        article["text"]
    ).ids

    assert len(text_ids) == row["full_text_tokens"]
    assert CORPUS_BOUNDARY_ID not in text_ids

    included_ids = np.asarray(
        text_ids[: row["included_text_tokens"]],
        dtype=TOKEN_DTYPE,
    )
    reconstruction_hasher.update(
        included_ids.tobytes(order="C")
    )
    reconstructed_token_count += len(included_ids)

    if row["boundary_included"]:
        reconstruction_hasher.update(boundary_bytes)
        reconstructed_token_count += 1

reconstructed_sha256 = reconstruction_hasher.hexdigest()

assert reconstructed_token_count == CORPUS_TOKEN_BUDGET
assert reconstructed_sha256 == corpus_token_sha256

print(
    "✓ Manifest reconstruction reproduced exactly "
    f"{reconstructed_token_count:,} tokens."
)
print(f"Reconstructed SHA-256: {reconstructed_sha256}")
print("✓ Reconstructed checksum matches the canonical corpus checksum.")


✓ Manifest reconstruction reproduced exactly 20,000,000 tokens.
Reconstructed SHA-256: 4101d5b18c38558a58110f54a161763186ab5318111366486ebbfa0a3fe584fa
✓ Reconstructed checksum matches the canonical corpus checksum.


## 36. Package the small canonical corpus artifacts

The repository artifacts are intentionally small:

```text
results/corpus/corpus_summary.json
results/corpus/corpus_manifest.jsonl
```

The 40 MB token binary is not included in this ZIP because the previous reconstruction test proves it can be regenerated exactly from the committed manifest and tokenizer artifact.


In [41]:
CORPUS_ZIP_BASE = Path("/content/corpus_manifest_bundle")

corpus_archive_path = Path(
    shutil.make_archive(
        str(CORPUS_ZIP_BASE),
        "zip",
        root_dir=CORPUS_ARTIFACT_DIR,
    )
)

corpus_archive_sha256 = hashlib.sha256(
    corpus_archive_path.read_bytes()
).hexdigest()

print(f"✓ Created corpus artifact bundle: {corpus_archive_path}")
print(f"Bundle size: {corpus_archive_path.stat().st_size:,} bytes")
print(f"Bundle SHA-256: {corpus_archive_sha256}")


✓ Created corpus artifact bundle: /content/corpus_manifest_bundle.zip
Bundle size: 183,638 bytes
Bundle SHA-256: 87878aed9d97c60291d03f634133765485a46dd8530101db517f6430fc549918


## 37. Download the corpus manifest bundle

Attach this ZIP together with the executed notebook when you return here.

The ZIP contains only the summary and manifest. The exact 40 MB token stream is identified by its canonical SHA-256 and has already been reconstructed from those artifacts inside the notebook.


In [42]:
from google.colab import files

files.download(str(corpus_archive_path))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Important upcoming packing issue: the 256-token tail

The exact corpus has **20,000,000 tokens**.

With a context length of 512:

`20,000,000 mod 512 = 256`

Chunk 6 deliberately does **not** discard those 256 tokens.

Corpus construction and training-example packing are different contracts:

- **this chunk** defines which 20,000,000 tokens belong to the controlled corpus;
- a later training-data chunk must explicitly define how those tokens become model input/target sequences.

We will not silently turn the experiment into 19,999,744 tokens just because that number divides evenly by 512.


### Pause here

Chunk 6 is the **exact-corpus construction gate**.

Before proceeding, we should have observed:

- deterministic permutation from seed **42**;
- exactly **20,000,000** tokens;
- `uint16` raw size exactly **40,000,000 bytes**;
- explicit boundary-token count matching the manifest;
- only the final selected article sequence potentially truncated;
- canonical token-stream SHA-256;
- ordered `corpus_manifest.jsonl`;
- `corpus_summary.json`;
- manifest-driven reconstruction reproducing the same **20,000,000 tokens and SHA-256**;
- no validation/test tokens in the model-training corpus;
- the 256-token context-length tail preserved rather than silently discarded.

After this passes, Notebook 02's core tokenizer/corpus construction work is effectively complete. At that point we should perform the **tokenizer-phase documentation sync** required by D-050, including:

- D-055 — special-token contract and observed evidence;
- D-056 — `min_frequency=2` stopping rule and observed 16,127 merges;
- the persisted tokenizer checksum/evidence;
- the exact 20M corpus construction evidence and manifest policy;
- the unresolved 512-token packing-tail decision for the later training pipeline.

Only after that documentation checkpoint should we transition toward Notebook 03 / model architecture.
